# Step 1 — Fluorescence intensity inside 3D masks, per cell and z slice

For every cell z-stack, this notebook takes a 3D mask or probability map, cleans it, optionally reduces it to a shell under the cell surface (for example the cortex), and measures the background-subtracted intensity of each channel inside it, slice by slice. It writes one folder per cell. Step 2 (`2_compare_conditions.ipynb`) reads these folders.

**Inputs** (same `<condition>/<experiment>/` sub-folders in both)
```
RAW_DIR/<condition>/<experiment>/<cell>.tif                  z-stack, axes (z, c, y, x), or (z, y, x) for one channel
MASK_DIR/<condition>/<experiment>/<cell><MASK_SUFFIX>        3D mask or probability map, axes (z, y, x)
```

**Outputs** in `OUTPUT_DIR/<condition>/<experiment>/<cell>/`
- `analysis_mask.tif`: the 3D mask used for the measurement (uint8, 0/255), same size as the z-stack
- `per_z.csv`: for each z slice, the number of mask pixels and the mean and SD of each background-subtracted channel inside the mask
- `qc_montage.png`: one channel, the cleaned foreground and the mask overlay at evenly spaced z slices

Set the folders, channels and mask settings in section 2, then run all cells from top to bottom. By default, cells that already have all three output files are skipped.

## 1 — Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from scipy.ndimage import binary_fill_holes, distance_transform_edt, label

## 2 — Settings

In [ ]:
# ── Folders ──────────────────────────────────────────────────────────────────
RAW_DIR    = Path('data/1_Raw')
MASK_DIR   = Path('data/2_Segmentations')
OUTPUT_DIR = Path('data/3_Processed')

# Set True to recompute cells that already have all output files
OVERWRITE = False

# ── Mask file ────────────────────────────────────────────────────────────────
MASK_SUFFIX = '_foreground_pmaps.tif'   # file name after the cell name
GATE_SUFFIX = None                      # e.g. '_boundary_pmaps.tif' to also restrict the mask to a second map

# ── Channels: name -> index in the z-stack ───────────────────────────────────
# The names become the column prefixes in per_z.csv
CHANNELS = {'channel_1': 0, 'channel_2': 1, 'channel_3': 2}
QC_CHANNEL = 'channel_1'   # channel shown in the QC montage

# ── Mask settings ────────────────────────────────────────────────────────────
MASK_THRESHOLD       = 0.5     # probability maps (0-1) and binary masks (0/1 or 0/255) both work
MIN_COMPONENT_VOXELS = 5_000   # 3D foreground objects smaller than this are removed
SHELL_THICKNESS_PX   = 10      # shell thickness, inward from the cell surface (pixels); None = the whole mask

# ── QC montage ───────────────────────────────────────────────────────────────
QC_N_SLICES = 4   # number of evenly spaced z slices shown per cell

OUTPUT_FILES = ('analysis_mask.tif', 'per_z.csv', 'qc_montage.png')

assert RAW_DIR.is_dir(), f'Raw folder not found: {RAW_DIR.resolve()}'
assert MASK_DIR.is_dir(), f'Mask folder not found: {MASK_DIR.resolve()}'
assert QC_CHANNEL in CHANNELS, f'QC_CHANNEL must be one of {list(CHANNELS)}'

## 3 — Functions

In [ ]:
def load_mask(path, threshold):
    # Load a mask or probability map and return a boolean (z, y, x) mask
    return tifffile.imread(str(path)) > threshold


def remove_small_components_3d(mask, min_voxels):
    # Remove 3D connected components smaller than min_voxels
    labelled, _ = label(mask)
    sizes = np.bincount(labelled.ravel())
    sizes[0] = 0
    keep = np.where(sizes >= min_voxels)[0]
    return np.isin(labelled, keep)


def clean_foreground(fg, min_voxels):
    # Fill holes in each z slice, then remove small 3D debris
    filled = np.zeros_like(fg, dtype=bool)
    for z in range(fg.shape[0]):
        filled[z] = binary_fill_holes(fg[z])
    return remove_small_components_3d(filled, min_voxels)


def build_shell(fg_clean, thickness_px):
    # Shell of uniform thickness inside the cell surface, from a 2D distance transform per z slice
    shell = np.zeros_like(fg_clean, dtype=bool)
    for z in range(fg_clean.shape[0]):
        sl = fg_clean[z]
        if not sl.any():
            continue
        dist = distance_transform_edt(sl)
        shell[z] = sl & (dist <= thickness_px)
    return shell


def background_from_fg(fg_mask, channel_data):
    # Mean intensity of all voxels outside the cleaned foreground (whole stack)
    return np.mean(channel_data[~fg_mask])


def quantify_channels_per_z(channels, mask):
    # channels: dict name -> (z, y, x) array; mask: (z, y, x) bool
    records = []
    for z in range(mask.shape[0]):
        mask_z = mask[z]
        row = {'z': z, 'n_pixels': int(mask_z.sum())}
        for ch_name, vol in channels.items():
            pixels = vol[z][mask_z]
            row[f'{ch_name}_mean'] = float(pixels.mean()) if pixels.size else np.nan
            row[f'{ch_name}_std']  = float(pixels.std())  if pixels.size else np.nan
        records.append(row)
    return pd.DataFrame(records)


def make_qc_montage(channel_img, mask, fg_clean, z_slices, channel_name, out_path):
    # One row per z slice: channel | cleaned foreground | mask (green) over channel
    fig, axes = plt.subplots(len(z_slices), 3, figsize=(10, 3 * len(z_slices)))
    if len(z_slices) == 1:
        axes = [axes]
    for row, z in enumerate(z_slices):
        ax_a, ax_f, ax_s = axes[row]
        ax_a.imshow(channel_img[z], cmap='gray')
        ax_a.set_title(f'Z={z}  {channel_name}', fontsize=7); ax_a.axis('off')
        ax_f.imshow(fg_clean[z], cmap='gray')
        ax_f.set_title(f'Z={z}  Cleaned FG', fontsize=7); ax_f.axis('off')
        rgb = np.stack([channel_img[z]] * 3, axis=-1).astype(float)
        rgb = rgb / (rgb.max() + 1e-9)
        rgb[mask[z]] = [0, 1, 0]
        ax_s.imshow(rgb)
        ax_s.set_title(f'Z={z}  Mask', fontsize=7); ax_s.axis('off')
    plt.suptitle(out_path.parent.name, fontsize=8, y=1.01)
    plt.tight_layout()
    fig.savefig(str(out_path), dpi=120, bbox_inches='tight')
    plt.close(fig)


def process_cell(job):
    # Build the mask for one cell and write its three output files
    raw = tifffile.imread(str(job['raw'])).astype(np.float32)
    if raw.ndim == 3:
        raw = raw[:, np.newaxis]   # one channel: (z, y, x) -> (z, 1, y, x)
    channels = {name: raw[:, index] for name, index in CHANNELS.items()}

    fg_raw = load_mask(job['mask'], MASK_THRESHOLD)
    if fg_raw.shape != channels[QC_CHANNEL].shape:
        raise ValueError(f'mask shape {fg_raw.shape} does not match z-stack shape {channels[QC_CHANNEL].shape}')
    fg_clean = clean_foreground(fg_raw, MIN_COMPONENT_VOXELS)
    mask = fg_clean if SHELL_THICKNESS_PX is None else build_shell(fg_clean, SHELL_THICKNESS_PX)
    if GATE_SUFFIX:
        mask = mask & load_mask(job['gate'], MASK_THRESHOLD)

    job['out'].mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(str(job['out'] / 'analysis_mask.tif'), (mask * 255).astype(np.uint8))

    # Background: mean outside the cleaned foreground, subtracted from each channel
    for ch in channels.values():
        ch -= background_from_fg(fg_clean, ch)

    df = quantify_channels_per_z(channels, mask)
    df.insert(0, 'cell', job['cell'])
    df.to_csv(job['out'] / 'per_z.csv', index=False)

    n_z = raw.shape[0]
    z_slices = np.linspace(0, n_z - 1, min(QC_N_SLICES, n_z), dtype=int).tolist()
    make_qc_montage(channels[QC_CHANNEL], mask, fg_clean, z_slices, QC_CHANNEL, job['out'] / 'qc_montage.png')
    return n_z, int(mask.sum())

## 4 — Find cells

In [ ]:
jobs, missing = [], []
for raw_path in sorted(RAW_DIR.glob('*/*/*.tif')):
    rel_dir = raw_path.parent.relative_to(RAW_DIR)       # <condition>/<experiment>
    cell = raw_path.stem
    job = {'cell': cell, 'condition': rel_dir.parts[0], 'experiment': rel_dir.parts[1],
           'raw': raw_path,
           'mask': MASK_DIR / rel_dir / (cell + MASK_SUFFIX),
           'gate': MASK_DIR / rel_dir / (cell + GATE_SUFFIX) if GATE_SUFFIX else None,
           'out': OUTPUT_DIR / rel_dir / cell}
    if job['mask'].exists() and (not GATE_SUFFIX or job['gate'].exists()):
        jobs.append(job)
    else:
        missing.append(cell)

assert jobs, f'No cells found: expected {RAW_DIR}/<condition>/<experiment>/<cell>.tif with a matching mask in {MASK_DIR}'
jobs_df = pd.DataFrame(jobs)
print(f'{len(jobs)} cells with masks')
print(jobs_df.groupby(['condition', 'experiment']).size().to_string())
if missing:
    print(f'\n{len(missing)} cells without masks (not processed):', *missing, sep='\n  ')

## 5 — Process all cells

In [ ]:
done, skipped, failed = 0, 0, []
for i, job in enumerate(jobs, 1):
    if not OVERWRITE and all((job['out'] / f).exists() for f in OUTPUT_FILES):
        skipped += 1
        continue
    try:
        n_z, n_mask = process_cell(job)
        done += 1
        print(f'[{i}/{len(jobs)}] {job["cell"]}: {n_z} z slices, {n_mask:,} mask voxels')
    except Exception as e:
        failed.append(job['cell'])
        print(f'[{i}/{len(jobs)}] {job["cell"]}: ERROR {e}')

print(f'\nProcessed: {done}   Skipped (outputs already present): {skipped}   Failed: {len(failed)}')
for cell in failed:
    print('  failed:', cell)

## 6 — Check outputs

In [ ]:
incomplete = [j['cell'] for j in jobs if not all((j['out'] / f).exists() for f in OUTPUT_FILES)]
print(f'{len(jobs) - len(incomplete)}/{len(jobs)} cells have all output files in {OUTPUT_DIR}')
for cell in incomplete:
    print('  incomplete:', cell)